# WorldFloodsV2 torch Dataset

* **Last Modified**: 31-07-2025
* **Authors**: Gonzalo Mateo-García (from Ruben Cartuyvels feedback)
---

> E. Portalés-Julià, G. Mateo-García, C. Purcell, and L. Gómez-Chova [Global flood extent segmentation in optical satellite images](https://www.nature.com/articles/s41598-023-47595-7). _Scientific Reports 13, 20316_ (2023). DOI: 10.1038/s41598-023-47595-7.


This example shows how to load the *WorldFloodsv2* dataset with a pytorch Dataset for training or inference purposes.


## Step 1: Download the v2 data from Hugging-Face 🤗

The WorldFloods v2 dataset is stored in Hugging-Face in the repository: [isp-uv-es/WorldFloodsv2](https://huggingface.co/datasets/isp-uv-es/WorldFloodsv2/). 

To download the full dataset (~76GB) run:

```
huggingface-cli download --cache-dir /path/to/cachedir --local-dir /path/to/localdir/WorldFloodsv2 --repo-type dataset isp-uv-es/WorldFloodsv2
```

## Step 2: Load the data with the `Dataset` on `ml4floods`

In [4]:
from ml4floods.models.config_setup import get_default_config
from ml4floods.models.dataset_setup import get_dataset
from typing import Any, Dict
import pandas as pd
import json
import os
import ml4floods


# Change accordingly!
DATASET_PATH = "/path/to/localdir/WorldFloodsv2"

CONFIG_PATH = os.path.join(os.path.dirname(ml4floods.__file__),"models/configurations/worldfloods_template_v2.json")

# Set this to the path of the metadata CSV from huggingface
CSV_PATH = os.path.join(DATASET_PATH,"dataset_metadata.csv")

# Point this to the root of the dataset on the mounted bucket
JSON_PATH = os.path.join(DATASET_PATH, "train_test_split_from_csv.json")


def convert_metadata_csv_to_json() -> None:
    out: Dict[str, Any] = {}
    modalities = ["S2", "gt"]
    csv = pd.read_csv(CSV_PATH)

    for split in csv.split.unique():
        out[split] = {}
        files = csv[csv.split == split]["event id"]
        for mod in modalities:
            out[split][mod] = [
                os.path.join(DATASET_PATH, split, mod, f"{fn}.tif")
                for fn in files.to_list()
            ]

    with open(JSON_PATH, "w") as f:
        json.dump(out, f, indent=2)


convert_metadata_csv_to_json()
config = get_default_config(CONFIG_PATH)

config.data_params.loader_type = "local"
config.data_params.bucket_id = None
config.data_params.path_to_splits = DATASET_PATH
config.data_params["download"] = {
    "train": False,
    "val": False,
    "test": False,
}
config.data_params.train_test_split_file = JSON_PATH

dm = get_dataset(config.data_params)
dm.prepare_data()
train_dl = dm.train_dataloader()
for batch in train_dl:
    print(batch)
    break


AssertionError: File input: ../../../WorldFloodsv2/train/S2/01042016_Choctawhatchee_River_near_Bellwood_AL.tif does not exists

## Licence
The ML4Floods package is published under a [GNU Lesser GPL v3 licence](https://www.gnu.org/licenses/lgpl-3.0.en.html)

The *WorldFloods* database and all pre-trained models are released under a [Creative Commons non-commercial licence](https://creativecommons.org/licenses/by-nc/4.0/legalcode.txt). For using the models in comercial pipelines written consent by the authors must be provided.

The Ml4Floods notebooks and docs are released under a [Creative Commons non-commercial licence](https://creativecommons.org/licenses/by-nc/4.0/legalcode.txt).

 If you find this work useful please cite:
```
@article{portales-julia_global_2023,
	title = {Global flood extent segmentation in optical satellite images},
	volume = {13},
	issn = {2045-2322},
	doi = {10.1038/s41598-023-47595-7},
	number = {1},
	urldate = {2023-11-30},
	journal = {Scientific Reports},
	author = {Portalés-Julià, Enrique and Mateo-García, Gonzalo and Purcell, Cormac and Gómez-Chova, Luis},
	month = nov,
	year = {2023},
	pages = {20316},
}
```

## Acknowledgments

This research has been supported by the DEEPCLOUD project (PID2019-109026RB-I00) funded by the Spanish Ministry of Science and Innovation (MCIN/AEI/10.13039/501100011033) and the European Union (NextGenerationEU).

<img width="300" title="DEEPCLOUD project (PID2019-109026RB-I00, University of Valencia) funded by MCIN/AEI/10.13039/501100011033." alt="DEEPCLOUD project (PID2019-109026RB-I00, University of Valencia) funded by MCIN/AEI/10.13039/501100011033." src="https://www.uv.es/chovago/logos/logoMICIN.jpg">